<a href="https://colab.research.google.com/github/huxd2334/DR-benchmark-/blob/CAnet/9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch
print(torch.cuda.is_available())  # Phải là True


True


In [ ]:
from __future__ import division
import os
import random
import shutil
import time
import argparse
import math
import numpy as np
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.nn.parallel
import torch.optim
import torch.utils.data
import torchvision.transforms as transforms
import torchvision.models as models
from torch.nn import init
from math import pi, cos
from skimage.transform import resize
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import label_binarize
try:
    from tensorboardX import SummaryWriter
except ImportError:
    import os
    os.system('pip install tensorboardX')
    from tensorboardX import SummaryWriter


2.6.0+cu124
0.21.0+cu124


In [ ]:
!git clone https://github.com/xmengli/CANet.git

Cloning into 'CANet'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 195 (delta 38), reused 29 (delta 29), pack-reused 147 (from 1)
Receiving objects: 100% (195/195), 229.38 KiB | 25.49 MiB/s, done.
Resolving deltas: 100% (102/102), done.


In [6]:
script_path = "/content/CANet/messidor_scripts/train_fold.sh"

# Đọc nội dung file shell script
with open(script_path, 'r') as file:
    lines = file.readlines()

modified_lines = []
for line in lines:
    # Sửa đúng đường dẫn python
    if "python" in line and "baseline.py" in line:
        line = line.replace("python baseline.py", "python /content/CANet/baseline.py")
        line = line.replace("python3 baseline.py", "python3 /content/CANet/baseline.py")
    elif "baseline.py" in line:
        # Nếu đã là absolute path thì không sửa nữa
        if "/content/CANet/baseline.py" not in line:
            line = line.replace("baseline.py", "/content/CANet/baseline.py")

    # Chỉ sửa nếu chưa có "/content"
    if "cd .." in line:
        line = "cd /content/CANet\n"
    if "./data/" in line:
        line = line.replace("./data/", "/content/CANet/data/")

    modified_lines.append(line)

# Ghi lại
with open(script_path, 'w') as file:
    file.writelines(modified_lines)


In [7]:
import os

python_script_path = "/content/CANet/baseline.py"

# Đọc nội dung file gốc
with open(python_script_path, 'r') as file:
    lines = file.readlines()

modified_lines = []
device_handled = False
pretrain_logic_injected = False

def update_device_handling(line):
    """Thêm logic xử lý thiết bị vào mã."""
    return """
    if torch.cuda.is_available() and args.gpu is not None:
        print(f"Use GPU: {args.gpu} for training")
        device = torch.device(f"cuda:{args.gpu}")
    else:
        print("⚠️ CUDA not available or GPU ID not specified. Using CPU.")
        device = torch.device("cpu")

    model = model.to(device)
"""

for line in lines:
    # Xóa dòng import scipy.misc
    if "import scipy.misc" in line:
        continue

    # Xử lý đường dẫn cho mô hình pretrained
    if "torch.load(pretrain_path" in line and not pretrain_logic_injected:
        indent = " " * (len(line) - len(line.lstrip()))
        pretrain_path = "pretrain/resnet50-19c8e357.pth"  # Cập nhật đường dẫn nếu cần
        full_pretrain_path = os.path.join("/content/CANet", pretrain_path)
        modified_lines.append(f'{indent}full_pretrain_path = "{full_pretrain_path}"\n')
        line = line.replace("torch.load(pretrain_path", "torch.load(full_pretrain_path")
        if "weights_only=False" not in line:
            line = line.replace(")", ", weights_only=False)")
        pretrain_logic_injected = True

    # Thêm logic xử lý thiết bị nếu chưa được thêm
    if "torch.cuda.set_device" in line and not device_handled:
        modified_lines.append(update_device_handling(line))
        device_handled = True
        continue  # Bỏ qua dòng gốc

    # Thay đổi .cuda(...) thành .to(device)
    line = line.replace(".cuda(args.gpu, non_blocking=True)", ".to(device)")
    line = line.replace(".cuda(args.gpu, non_blocking = True)", ".to(device)")
    line = line.replace(".cuda(args.gpu)", ".to(device)")
    line = line.replace(".cuda()", ".to(device)")

    modified_lines.append(line)

# Ghi lại file đã sửa
with open(python_script_path, 'w') as file:
    file.writelines(modified_lines)

print("✅ baseline.py đã được khôi phục và sửa đúng hoàn toàn, không còn lỗi lặp đường dẫn.")

✅ baseline.py đã được khôi phục và sửa đúng hoàn toàn, không còn lỗi lặp đường dẫn.


In [ ]:
!ls -lh /content/CANet/data/Base11/

total 956M
-rw-r--r-- 1 root root 9.6M Oct 19  2005 20051019_38557_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_43808_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_43832_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_43882_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_43906_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44261_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44284_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44338_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44349_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44400_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44431_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44598_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44636_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2005 20051020_44692_0100_PP.tif
-rw-r--r-- 1 root root 9.6M Oct 20  2

In [8]:
!touch /content/CANet/datasets/__init__.py

In [9]:

# Thử import trực tiếp file
import importlib.util

spec = importlib.util.spec_from_file_location("missidor", "/content/CANet/datasets/missidor.py")
missidor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(missidor)

# Gọi thử traindataset nếu có
if hasattr(missidor, "traindataset"):
    print("✅ Imported traindataset thành công!")
else:
    print("⚠️ Không tìm thấy traindataset trong missidor.py")


✅ Imported traindataset thành công!


In [4]:
!for i in 11 12 13 14; do \
  echo "🔍 Giải nén nếu chưa tồn tại: Base$i.zip..."; \
  unzip -q -o /content/CANet/data/Base${i}.zip -d /content/CANet/data/Base${i}/; \
done

🔍 Giải nén nếu chưa tồn tại: Base11.zip...
🔍 Giải nén nếu chưa tồn tại: Base12.zip...
🔍 Giải nén nếu chưa tồn tại: Base13.zip...
🔍 Giải nén nếu chưa tồn tại: Base14.zip...


In [16]:
!sh /content/CANet/messidor_scripts/train_fold.sh

Use GPU: 0 for training
==> Load pretrained model
⚠️ CUDA not available or GPU ID not specified. Using CPU.
    Total params: 29.03M
Traceback (most recent call last):
  File "/content/CANet/baseline.py", line 629, in <module>
    main()
  File "/content/CANet/baseline.py", line 115, in main
    main_worker(args.gpu, args)
  File "/content/CANet/baseline.py", line 270, in main_worker
    val_dataset = traindataset(root=args.data, mode = 'val',
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/CANet/datasets/missidor.py", line 40, in __init__
    idx = np.loadtxt(self.root + "/10fold/"+str(args.fold_name)+".txt", dtype=int)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/numpy/lib/_npyio_impl.py", line 1381, in loadtxt
    arr = _read(fname, dtype=dtype, comment=comment, delimiter=delimiter,
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/

In [17]:
!sh /content/CANet/messidor_scripts/eval_fold.sh

Use GPU: 0 for training
==> Load pretrained model
⚠️ CUDA not available or GPU ID not specified. Using CPU.
    Total params: 29.03M
=> no checkpoint found at 'exp/MESSIDOR//multi_CAN_lamda25_noshare_wpre_simpleaug_10fold1_1000/model_converge.pth.tar'
Use GPU: 0 for training
==> Load pretrained model
⚠️ CUDA not available or GPU ID not specified. Using CPU.
    Total params: 29.03M
=> no checkpoint found at 'exp/MESSIDOR//multi_CAN_lamda25_noshare_wpre_simpleaug_10fold2_1000/model_converge.pth.tar'
Use GPU: 0 for training
==> Load pretrained model
⚠️ CUDA not available or GPU ID not specified. Using CPU.
    Total params: 29.03M
=> no checkpoint found at 'exp/MESSIDOR//multi_CAN_lamda25_noshare_wpre_simpleaug_10fold3_1000/model_converge.pth.tar'
Use GPU: 0 for training
==> Load pretrained model
⚠️ CUDA not available or GPU ID not specified. Using CPU.
    Total params: 29.03M
=> no checkpoint found at 'exp/MESSIDOR//multi_CAN_lamda25_noshare_wpre_simpleaug_10fold4_1000/model_converge.p